In [1]:
import pandas as pd
import gc

In [2]:
chunksize = 100_000  # 每次讀10萬筆

# 建立空 list 收集符合條件的資料
filtered_chunks = []

for chunk in pd.read_csv("title.basics.tsv.gz", sep='\t', dtype=str, na_values='\\N', chunksize=chunksize):
    chunk = chunk[(chunk['titleType'] == 'movie') & (chunk['startYear'].notna())]
    
    # 有些 startYear 非數字，轉換時需處理
    chunk = chunk[chunk['startYear'].str.isnumeric()]
    chunk['startYear'] = chunk['startYear'].astype(int)
    filtered_chunks.append(chunk)

# 合併所有過濾後的 chunk
basics_filtered = pd.concat(filtered_chunks, ignore_index=True)
print(f"載入後共有 {len(basics_filtered)} 筆符合條件的電影")

載入後共有 641035 筆符合條件的電影


In [3]:
basics_filtered

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000009,movie,Miss Jerry,Miss Jerry,0,1894,NaN,45,Romance
1,tt0000147,movie,The Corbett-Fitzsimmons Fight,The Corbett-Fitzsimmons Fight,0,1897,NaN,100,"Documentary,News,Sport"
2,tt0000502,movie,Bohemios,Bohemios,0,1905,NaN,100,NaN
3,tt0000574,movie,The Story of the Kelly Gang,The Story of the Kelly Gang,0,1906,NaN,70,"Action,Adventure,Biography"
4,tt0000591,movie,The Prodigal Son,L'enfant prodigue,0,1907,NaN,90,Drama
...,...,...,...,...,...,...,...,...,...
641030,tt9916622,movie,Rodolpho Teóphilo - O Legado de um Pioneiro,Rodolpho Teóphilo - O Legado de um Pioneiro,0,2015,NaN,57,Documentary
641031,tt9916680,movie,De la ilusión al desconcierto: cine colombiano...,De la ilusión al desconcierto: cine colombiano...,0,2007,NaN,100,Documentary
641032,tt9916706,movie,Dankyavar Danka,Dankyavar Danka,0,2013,NaN,NaN,Comedy
641033,tt9916730,movie,6 Gunn,6 Gunn,0,2017,NaN,116,Drama


In [4]:
# 載入評分資料
ratings = pd.read_csv("title.ratings.tsv.gz", sep='\t', dtype={'tconst': str})

In [5]:
ratings

,tconst,averageRating,numVotes
0,tt0000001,5.7,2225
1,tt0000002,5.4,324
2,tt0000003,6.4,2371
3,tt0000004,5.0,200
4,tt0000005,6.2,3083
...,...,...,...
1705280,tt9916846,5.3,7
1705281,tt9916848,5.2,7
1705282,tt9916850,6.0,7
1705283,tt9916852,5.7,7


In [6]:
# 只取需要的欄位
basics_cols = ['tconst', 'primaryTitle', 'originalTitle', 'startYear', 'genres']

# left join，把電影資訊串進來
result = pd.merge(
    ratings,
    basics_filtered[basics_cols],
    on='tconst',
    how='inner'  # 只保留有評分的電影
)

# 顯示結果
result

,tconst,averageRating,numVotes,primaryTitle,originalTitle,startYear,genres
0,tt0000009,5.3,237,Miss Jerry,Miss Jerry,1894,Romance
1,tt0000147,5.3,610,The Corbett-Fitzsimmons Fight,The Corbett-Fitzsimmons Fight,1897,"Documentary,News,Sport"
2,tt0000502,3.6,27,Bohemios,Bohemios,1905,NaN
3,tt0000574,6.0,1083,The Story of the Kelly Gang,The Story of the Kelly Gang,1906,"Action,Adventure,Biography"
4,tt0000591,4.8,40,The Prodigal Son,L'enfant prodigue,1907,Drama
...,...,...,...,...,...,...,...
349237,tt9916362,6.4,6336,Coven,Akelarre,2020,"Drama,History,Horror"
349238,tt9916428,4.7,25,The Secret of China,Hong xing zhao yao Zhong guo,2019,"Adventure,History,War"
349239,tt9916538,7.6,12,I'll Take My Heart Again,Kuambil Lagi Hatiku,2019,Drama
349240,tt9916706,6.5,13,Dankyavar Danka,Dankyavar Danka,2013,Comedy


In [7]:
del ratings
del basics_filtered
gc.collect()

0

In [8]:
result = result[(result['numVotes']>1500) & (result['averageRating']>6.5)]
result = result.drop(columns=['tconst', 'originalTitle'])
result

,averageRating,numVotes,primaryTitle,startYear,genres
94,7.1,4225,Dante's Inferno,1911,"Adventure,Drama,Fantasy"
142,6.9,2746,Fantômas: In the Shadow of the Guillotine,1913,"Crime,Drama"
155,7.0,1623,Ingeborg Holm,1913,Drama
158,6.9,1864,Fantomas: The Man in Black,1913,"Crime,Drama"
229,7.1,4381,Cabiria,1914,"Adventure,Drama,History"
...,...,...,...,...,...
349103,6.6,3462,Mogul Mowgli,2020,"Drama,Music"
349124,6.8,2556,Min pappa Marianne,2020,"Comedy,Drama"
349162,8.4,52364,Kaithi,2019,"Action,Crime,Thriller"
349165,7.0,5398,Herself,2020,Drama


In [9]:
result.to_csv("raw_imdb.csv", index=False)